# 🏥 Sổ Tay 06: Kiểm Tra Nhãn Ground Truth & Nhận Diện AI Cho Các Ca Đạo Cụ Y Tế
## Medical Props & Object Cases Audit: Ground Truth Validation vs. AI Detection
---
Sổ tay này chuyên biệt dành cho việc kiểm tra chất lượng nhãn Ground Truth (được dán tự động qua lưới 128 tia Multi-Raycast vật lý) và đối soát khả năng suy luận của các mô hình AI (`YOLOv8n`, `YOLO11n`, `YOLO26n`) trên **27 Ca Đạo Cụ & Vật Thể Y Tế**:

* 🛡️ **Nhóm $U_3$ (12 ca)**: Bệnh nhân / Nhân viên bị che khuất bởi đạo cụ y tế (Cột truyền dịch, Xe monitor, Kiosk bấm số, Cây nước, Cột khử khuẩn, Xe cáng, Standee, Quầy lễ tân, Vách rèm, Xe tiêm Inox, Góc tường, Ngưỡng cửa).
* 🪤 **Nhóm $U_8$ (8 ca)**: Bẫy đạo cụ rỗng $0\text{ GT}$ (Hành lang rỗng, Cột truyền dịch rỗng, Cột khử khuẩn rỗng, Kiosk rỗng, Xe monitor rỗng, Xe cáng rỗng, Standee 2D in hình bác sĩ, Xe lăn rỗng). *Mục tiêu: Đánh giá nguy cơ Báo động giả (False Alarms) gây hiện tượng phanh giật cục trên Robot.*
* 🦽 **Nhóm $U_4$ (7 ca)**: Phương tiện hỗ trợ di chuyển (Xe lăn các góc $0^\circ, 45^\circ, 90^\circ$, Xe lăn có người đẩy, Xe cáng di động, Giường bệnh nội trú, Xe lăn gấp gọn).

---
### 📌 Hướng dẫn sử dụng:
1. Chạy các ô thiết lập môi trường và nạp chỉ mục dữ liệu.
2. Sử dụng công cụ tương tác **Interactive Prop Inspector** (`ipywidgets`) để chọn nhanh ca kiểm thử, số khung hình, mô hình và ngưỡng tự tin.
3. Chạy phần **Batch Prop Trap Gallery** để ngắm nhìn toàn cảnh các bẫy đạo cụ rỗng dưới góc nhìn của AI.


In [ ]:
import os
import sys
import json
import time
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Thêm đường dẫn mã nguồn phát hiện
repo_root = Path("..").resolve() if Path("..").joinpath("sim_vision").exists() else Path(".").resolve()
sys.path.insert(0, str(repo_root / "sim_vision/python"))

from detector import YoloDetector

print(f"✅ Đã thiết lập môi trường. Thư mục gốc: {repo_root}")


In [ ]:
dataset_dir = repo_root / "sim_vision/data/vision_eval_unified"
img_dir = dataset_dir / "images"
lbl_dir = dataset_dir / "labels"
jsonl_file = dataset_dir / "annotations.jsonl"
manifest_file = dataset_dir / "manifest_unified.json"

# Kiểm tra sự tồn tại của tập dữ liệu
assert dataset_dir.exists(), f"Không tìm thấy thư mục: {dataset_dir}"
assert jsonl_file.exists(), f"Không tìm thấy annotations.jsonl: {jsonl_file}"

with open(manifest_file, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"📦 Dataset: {manifest.get('dataset_name')} (Phiên bản: {manifest.get('version')})")
print(f"📊 Tổng số frames: {manifest.get('total_frames')}, Tổng số ca: {manifest.get('total_cases')}")

# Nạp toàn bộ metadata annotations.jsonl
annotations_registry = {}
with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        annotations_registry[data["frame_id"]] = data

print(f"✅ Đã nạp chỉ mục {len(annotations_registry)} frames vào bộ nhớ tra cứu.")


In [ ]:
# Phân loại và lập danh mục chuyên biệt cho 27 CA ĐẠO CỤ & VẬT THỂ Y TẾ

PROP_GROUPS = {
    "U3_MedicalProps": {
        "title": "U3: Che Khuất Bởi Đạo Cụ Y Tế (12 Ca)",
        "cases": [
            ("U3.1", "Sau Cột Truyền Dịch Inox", "iv_stand"),
            ("U3.2", "Sau Xe Đẩy Monitor 1.2m", "vital_monitor"),
            ("U3.3", "Thao Tác Bấm Kiosk 1.55m", "queue_kiosk"),
            ("U3.4", "Lấy Nước Bên Cây Nóng Lạnh", "water_dispenser"),
            ("U3.5", "Rửa Tay Cột Khử Khuẩn", "sanitizer_stand"),
            ("U3.6", "Hộ Lý Đẩy Xe Cáng 2.1m", "stretcher_cart"),
            ("U3.7", "Đứng Sát Mép Standee", "info_standee"),
            ("U3.8", "Nhân Viên Sau Quầy Lễ Tân", "vital_monitor"),
            ("U3.9", "Khuất Sau Vách Rèm Y Tế", "privacy_screen"),
            ("U3.10", "Khuất Sau Xe Tiêm Inox", "medical_trolley"),
            ("U3.12", "Ló Sau Góc Tường Hành Lang", "privacy_screen"),
            ("U3.13", "Qua Ngưỡng Cửa Phòng 110", "info_standee")
        ]
    },
    "U8_EmptyTraps": {
        "title": "U8: Bẫy Đạo Cụ Rỗng - 0 Ground Truth (8 Ca)",
        "cases": [
            ("U8.1", "Hành Lang Rỗng Ban Đêm", "empty_corridor"),
            ("U8.2", "Bẫy Cột Truyền Dịch Rỗng", "empty_iv_stand"),
            ("U8.3", "Bẫy Cột Khử Khuẩn Rỗng", "empty_sanitizer_stand"),
            ("U8.4", "Bẫy Kiosk Bấm Số Rỗng", "empty_queue_kiosk"),
            ("U8.5", "Bẫy Xe Đẩy Monitor Rỗng", "empty_vital_monitor"),
            ("U8.6", "Bẫy Xe Cáng Cấp Cứu Rỗng", "empty_stretcher_cart"),
            ("U8.8", "Bẫy Standee 2D In Hình Y Tế", "empty_info_standee"),
            ("U8.10", "Bẫy Xe Lăn Rỗng", "empty_wheelchair")
        ]
    },
    "U4_WheelchairBed": {
        "title": "U4: Phương Tiện Hỗ Trợ & Xe Lăn (7 Ca)",
        "cases": [
            ("U4.1", "Xe Lăn Trực Diện 0°", "wheelchair_patient"),
            ("U4.2", "Xe Lăn Chéo Góc 45°", "wheelchair_patient"),
            ("U4.3", "Xe Lăn Nhìn Ngang 90°", "wheelchair_patient"),
            ("U4.5", "Cụ Già Ngồi Xe Lăn Có Người Đẩy", "pushed_wheelchair"),
            ("U4.6", "Xe Cáng Cứu Thương Di Động", "mobile_stretcher"),
            ("U4.7", "Giường Bệnh Nội Trú Có Người", "hospital_bed"),
            ("U4.8", "Xe Lăn Gấp Gọn Dựa Tường", "folded_wheelchair")
        ]
    }
}

total_prop_cases = sum(len(g["cases"]) for g in PROP_GROUPS.values())
print(f"🔍 Đã xác lập {total_prop_cases} ca kiểm thử đạo cụ chuyên sâu (Tổng cộng: {total_prop_cases * 18} frames).")


In [ ]:
# Hàm tính toán IoU 2D giữa hai bounding box dạng [x1, y1, x2, y2]
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interWidth = max(0.0, xB - xA)
    interHeight = max(0.0, yB - yA)
    interArea = interWidth * interHeight

    boxAArea = max(0.0, (boxA[2] - boxA[0]) * (boxA[3] - boxA[1]))
    boxBArea = max(0.0, (boxB[2] - boxB[0]) * (boxB[3] - boxB[1]))
    unionArea = boxAArea + boxBArea - interArea

    return interArea / unionArea if unionArea > 0.0 else 0.0

# Bộ nhớ đệm nạp các detector AI
_loaded_detectors = {}

def get_detector(model_name="yolov8n.pt", conf_thresh=0.25):
    key = (model_name, conf_thresh)
    if key not in _loaded_detectors:
        _loaded_detectors[key] = YoloDetector(model_name, conf=conf_thresh)
    return _loaded_detectors[key]

print("✅ Đã khởi tạo các hàm tiện ích kiểm thử IoU và Detector Cache.")


---
## 🔍 Hàm Thanh Tra Trực Quan Đạo Cụ: `inspect_prop_frame(...)`
Hàm này tải ảnh, trích xuất siêu dữ liệu Ground Truth (hộp bao nhìn thấy, độ che khuất, cự ly) và cho mô hình AI suy luận trực tiếp để đối chiếu 3 góc nhìn:
1. **Khung hình gốc & Ground Truth chuẩn xác**: Hộp xanh lá là vùng cơ thể thực sự lộ ra ngoài đạo cụ.
2. **Dự đoán của AI**: Hiển thị hộp bao nhận diện người, phát hiện báo động giả trên đạo cụ rỗng.
3. **Phân tích khớp nối (Overlay & Error Analysis)**: Đánh dấu xanh (TP), đỏ (FP - Phanh nhầm), hồng (FN - Bỏ sót).


In [ ]:
def inspect_prop_frame(case_id, frame_idx=1, model_name="yolov8n.pt", conf_thresh=0.25, show_plot=True):
    fid = f"{case_id.replace('.', '_')}_{frame_idx:04d}"
    img_path = img_dir / f"{fid}.jpg"
    
    if not img_path.exists():
        print(f"❌ Không tìm thấy file ảnh: {img_path}")
        return None
        
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    
    meta = annotations_registry.get(fid, {})
    name_vi = meta.get("name_vi", "Không rõ")
    group_name = meta.get("group_name", "Không rõ")
    is_hard_neg = meta.get("is_hard_negative", False) or case_id.startswith("U8")
    
    # Ground truth boxes
    gts = meta.get("ground_truth_annotations", [])
    gt_boxes = []
    for g in gts:
        b = g.get("bbox_xyxy_px", [])
        if len(b) == 4:
            gt_boxes.append({
                "box": b,
                "vis": g.get("visibility_ratio", 1.0),
                "is_occ": g.get("is_occluded", False),
                "actor": g.get("actor_name", "person")
            })
            
    # AI Detection
    detector = get_detector(model_name, conf_thresh)
    t0 = time.perf_counter()
    raw_dets, infer_ms = detector.detect(img_bgr)
    infer_time_ms = (time.perf_counter() - t0) * 1000.0
    
    pred_boxes = []
    for d in raw_dets:
        if d.get("class_id") == 0: # person
            px1, py1 = d["x1"] * w, d["y1"] * h
            px2, py2 = d["x2"] * w, d["y2"] * h
            pred_boxes.append({
                "box": [px1, py1, px2, py2],
                "conf": d.get("conf", d.get("confidence", 0.0))
            })
            
    # IoU Matching
    matched_gt = set()
    tp_boxes = []
    fp_boxes = []
    
    for pb in pred_boxes:
        best_iou = 0.0
        best_g_idx = -1
        for g_idx, gb in enumerate(gt_boxes):
            if g_idx in matched_gt:
                continue
            iou = compute_iou(pb["box"], gb["box"])
            if iou > best_iou:
                best_iou = iou
                best_g_idx = g_idx
                
        if best_iou >= 0.35 and best_g_idx != -1:
            matched_gt.add(best_g_idx)
            tp_boxes.append((pb, best_iou))
        else:
            fp_boxes.append(pb)
            
    fn_boxes = [gb for g_idx, gb in enumerate(gt_boxes) if g_idx not in matched_gt]
    
    # Render Plots
    if show_plot:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=110)
        
        # 1. GROUND TRUTH
        axes[0].imshow(img_rgb)
        axes[0].set_title(f"1. Ground Truth Thật (128-Raycast)\nGT Persons: {len(gt_boxes)}", fontsize=11, fontweight="bold", color="darkgreen")
        for gb in gt_boxes:
            bx = gb["box"]
            color = "lime" if gb["vis"] >= 0.5 else "orange"
            rect = patches.Rectangle((bx[0], bx[1]), bx[2]-bx[0], bx[3]-bx[1], linewidth=2.5, edgecolor=color, facecolor="none")
            axes[0].add_patch(rect)
            axes[0].text(bx[0], max(0, bx[1]-6), f"{gb['actor']} (Vis: {gb['vis']*100:.0f}%)", color="black", fontsize=8,
                         bbox=dict(facecolor=color, alpha=0.8, pad=1.5, edgecolor="none"))
        if is_hard_neg:
            axes[0].text(w/2, h/2, "BẪY ĐẠO CỤ RỖNG (0 GT)", color="white", fontsize=13, fontweight="bold", ha="center",
                         bbox=dict(facecolor="blue", alpha=0.7, pad=4))
        elif len(gt_boxes) == 0:
            axes[0].text(w/2, h/2, "CHE KHUẤT >80% (0 GT)", color="white", fontsize=13, fontweight="bold", ha="center",
                         bbox=dict(facecolor="dimgray", alpha=0.8, pad=4))
        axes[0].axis("off")
        
        # 2. AI PREDICTION
        axes[1].imshow(img_rgb)
        axes[1].set_title(f"2. Suy Luận AI ({model_name})\nDetections: {len(pred_boxes)} (conf>={conf_thresh})", fontsize=11, fontweight="bold", color="darkblue")
        for pb in pred_boxes:
            bx = pb["box"]
            is_trap_fp = is_hard_neg
            color = "red" if is_trap_fp else "deepskyblue"
            rect = patches.Rectangle((bx[0], bx[1]), bx[2]-bx[0], bx[3]-bx[1], linewidth=2.5, edgecolor=color, facecolor="none")
            axes[1].add_patch(rect)
            label_text = f"person {pb['conf']:.2f}"
            if is_trap_fp:
                label_text = f"⚠️ PHANH NHẦM! {pb['conf']:.2f}"
            axes[1].text(bx[0], max(0, bx[1]-6), label_text, color="white", fontsize=8, fontweight="bold",
                         bbox=dict(facecolor=color, alpha=0.8, pad=1.5, edgecolor="none"))
        axes[1].axis("off")
        
        # 3. OVERLAY ANALYSIS
        axes[2].imshow(img_rgb)
        axes[2].set_title(f"3. Thẩm Định Khớp Nối (Overlay)\nTP={len(tp_boxes)} | FP={len(fp_boxes)} | FN={len(fn_boxes)}", fontsize=11, fontweight="bold", color="purple")
        for pb, iou in tp_boxes:
            bx = pb["box"]
            rect = patches.Rectangle((bx[0], bx[1]), bx[2]-bx[0], bx[3]-bx[1], linewidth=2.5, edgecolor="lime", facecolor="none")
            axes[2].add_patch(rect)
            axes[2].text(bx[0], max(0, bx[1]-6), f"TP (IoU: {iou:.2f})", color="black", fontsize=8,
                         bbox=dict(facecolor="lime", alpha=0.8, pad=1.5, edgecolor="none"))
        for pb in fp_boxes:
            bx = pb["box"]
            rect = patches.Rectangle((bx[0], bx[1]), bx[2]-bx[0], bx[3]-bx[1], linewidth=2.5, edgecolor="red", facecolor="none", linestyle="--")
            axes[2].add_patch(rect)
            axes[2].text(bx[0], max(0, bx[1]-6), f"FP (Conf: {pb['conf']:.2f})", color="white", fontsize=8,
                         bbox=dict(facecolor="red", alpha=0.8, pad=1.5, edgecolor="none"))
        for gb in fn_boxes:
            bx = gb["box"]
            rect = patches.Rectangle((bx[0], bx[1]), bx[2]-bx[0], bx[3]-bx[1], linewidth=2.5, edgecolor="magenta", facecolor="none", linestyle=":")
            axes[2].add_patch(rect)
            axes[2].text(bx[0], bx[3]+12, f"FN (Bỏ Sót Vis: {gb['vis']*100:.0f}%)", color="white", fontsize=8,
                         bbox=dict(facecolor="magenta", alpha=0.8, pad=1.5, edgecolor="none"))
        axes[2].axis("off")
        
        plt.suptitle(f"[{case_id}] {name_vi} — Frame: {fid} | Cự ly: {meta.get('distance_m', '?')}m | Yaw: {meta.get('camera_yaw', '?')}° | Độ cao: {meta.get('camera_height', '?')}m",
                     fontsize=13, fontweight="bold", y=0.98)
        plt.tight_layout()
        plt.show()
        
    return {
        "frame_id": fid,
        "case_id": case_id,
        "name_vi": name_vi,
        "gt_count": len(gt_boxes),
        "pred_count": len(pred_boxes),
        "tp": len(tp_boxes),
        "fp": len(fp_boxes),
        "fn": len(fn_boxes),
        "is_hard_neg": is_hard_neg,
        "latency_ms": round(infer_time_ms, 2)
    }

print("✅ Đã định nghĩa hàm `inspect_prop_frame`. Sẵn sàng chạy thử nghiệm!")


---
## 🎛️ Bảng Điều Khiển Tương Tác (Interactive Prop Inspector Widget)
Sử dụng các thanh trượt và menu thả xuống bên dưới để kiểm tra bất kỳ khung hình nào của 27 ca đạo cụ:
* **Nhóm ca**: Chọn $U_3$ (Người sau đạo cụ), $U_8$ (Bẫy đạo cụ rỗng), $U_4$ (Xe lăn & Giường bệnh).
* **Ca kiểm thử**: Danh sách từng ca cụ thể.
* **Số thứ tự Frame**: $1 \to 18$.
* **Mô hình AI**: Lựa chọn giữa `yolov8n.pt`, `yolo11n.pt`, và `yolo26n.pt`.
* **Ngưỡng tự tin**: Điều chỉnh từ $0.10$ đến $0.80$.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Danh sách lựa chọn nhóm ca
group_dropdown = widgets.Dropdown(
    options=[(g_info["title"], g_key) for g_key, g_info in PROP_GROUPS.items()],
    value="U8_EmptyTraps",
    description="Nhóm Đạo Cụ:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

case_dropdown = widgets.Dropdown(
    options=[(f"[{cid}] {cname}", cid) for cid, cname, _ in PROP_GROUPS["U8_EmptyTraps"]["cases"]],
    value="U8.8",
    description="Ca Kiểm Thử:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

def update_case_options(*args):
    selected_group = group_dropdown.value
    case_dropdown.options = [(f"[{cid}] {cname}", cid) for cid, cname, _ in PROP_GROUPS[selected_group]["cases"]]
    case_dropdown.value = case_dropdown.options[0][1]

group_dropdown.observe(update_case_options, "value")

frame_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=18,
    step=1,
    description="Frame Số (1-18):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

model_dropdown = widgets.Dropdown(
    options=["yolov8n.pt", "yolo11n.pt", "yolo26n.pt"],
    value="yolov8n.pt",
    description="Mô Hình AI:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

conf_slider = widgets.FloatSlider(
    value=0.25,
    min=0.10,
    max=0.85,
    step=0.05,
    description="Ngưỡng Tự Tin (Conf):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

interactive_panel = widgets.interactive_output(
    lambda grp, case_id, frame_idx, model, conf: inspect_prop_frame(case_id, frame_idx, model, conf),
    {"grp": group_dropdown, "case_id": case_dropdown, "frame_idx": frame_slider, "model": model_dropdown, "conf": conf_slider}
)

controls_ui = widgets.VBox([
    widgets.HBox([group_dropdown, case_dropdown]),
    widgets.HBox([frame_slider, model_dropdown, conf_slider]),
    widgets.HTML("<hr style='margin: 8px 0;'>")
])

display(controls_ui, interactive_panel)


### 🧪 Ví Dụ Trực Quan Nhanh (Static Inspection Examples)
Dưới đây là 3 ca đạo cụ điển hình minh họa cho các trường phái kiểm thử khác nhau:
1. **`U8.8` (Bẫy Standee 2D)**: Đạo cụ không người nhưng AI nhận nhầm do thiếu chiều sâu 3D (Báo động giả phanh gấp).
2. **`U3.1` (Sau Cột Truyền Dịch)**: Multi-Raycast bóc tách chính xác phần cơ thể nhìn thấy.
3. **`U4.1` (Xe Lăn Trực Diện)**: Đối tượng yếu thế với hình thái ngồi thấp đặc thù y tế.


In [ ]:
# 1. Bẫy Standee 2D in hình y tế (U8.8 - 0 Ground Truth)
print("--- Ví dụ 1: Bẫy Standee 2D in hình y tế (U8.8) ---")
inspect_prop_frame("U8.8", frame_idx=1, model_name="yolov8n.pt", conf_thresh=0.25)

# 2. Sau Cột Truyền Dịch Inox (U3.1 - Frame 3)
print("--- Ví dụ 2: Sau Cột Truyền Dịch Inox (U3.1) ---")
inspect_prop_frame("U3.1", frame_idx=3, model_name="yolov8n.pt", conf_thresh=0.25)

# 3. Xe Lăn Trực Diện 0° (U4.1)
print("--- Ví dụ 3: Xe Lăn Trực Diện 0° (U4.1) ---")
inspect_prop_frame("U4.1", frame_idx=1, model_name="yolov8n.pt", conf_thresh=0.25)


---
## 🖼️ Thư Viện Toàn Cảnh: Khảo Sát 8 Bẫy Đạo Cụ Rỗng $U_8$ (0 Ground Truth)
Kiểm tra xem mô hình AI có bị đánh lừa bởi các đạo cụ y tế không người hay không:
* $U_{8.1}$: Hành lang vắng ban đêm
* $U_{8.2}$: Cọc truyền dịch inox rỗng
* $U_{8.3}$: Cột máy sát khuẩn rỗng
* $U_{8.4}$: Kiosk bấm số rỗng
* $U_{8.5}$: Xe đẩy monitor rỗng
* $U_{8.6}$: Xe cáng cấp cứu rỗng
* $U_{8.8}$: Standee 2D in hình bác sĩ/y tế (**Bẫy kinh điển**)
* $U_{8.10}$: Xe lăn rỗng dọc hành lang


In [ ]:
def render_u8_empty_traps_gallery(model_name="yolov8n.pt", conf_thresh=0.25, frame_step=1):
    u8_cases = PROP_GROUPS["U8_EmptyTraps"]["cases"]
    n_cases = len(u8_cases)
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10), dpi=110)
    axes = axes.flatten()
    
    detector = get_detector(model_name, conf_thresh)
    
    print(f"🚀 Đang render Thư Viện 8 Bẫy Đạo Cụ Rỗng ({model_name}, Conf={conf_thresh})...")
    for idx, (cid, cname, _) in enumerate(u8_cases):
        fid = f"{cid.replace('.', '_')}_{frame_step:04d}"
        img_p = img_dir / f"{fid}.jpg"
        ax = axes[idx]
        
        if not img_p.exists():
            ax.set_title(f"[{cid}] Thiếu ảnh", fontsize=10)
            ax.axis("off")
            continue
            
        img_bgr = cv2.imread(str(img_p))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]
        
        raw_dets, _ = detector.detect(img_bgr)
        preds = []
        for d in raw_dets:
            if d.get("class_id") == 0:
                px1, py1 = d["x1"] * w, d["y1"] * h
                px2, py2 = d["x2"] * w, d["y2"] * h
                preds.append({"box": [px1, py1, px2, py2], "conf": d.get("conf", d.get("confidence", 0.0))})
                
        ax.imshow(img_rgb)
        is_false_alarm = len(preds) > 0
        border_color = "red" if is_false_alarm else "green"
        status_text = f"⚠️ BÁO ĐỘNG GIẢ ({len(preds)})" if is_false_alarm else "✅ SẠCH (0 FP)"
        
        for pb in preds:
            bx = pb["box"]
            rect = patches.Rectangle((bx[0], bx[1]), bx[2]-bx[0], bx[3]-bx[1], linewidth=2.5, edgecolor="red", facecolor="none")
            ax.add_patch(rect)
            ax.text(bx[0], max(0, bx[1]-4), f"person {pb['conf']:.2f}", color="white", fontsize=8,
                    bbox=dict(facecolor="red", alpha=0.8, pad=1))
                    
        ax.set_title(f"[{cid}] {cname}\n{status_text}", fontsize=10, fontweight="bold", color=border_color)
        ax.axis("off")
        
        # Thêm khung viền màu cảnh báo
        for spine in ax.spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(3)
            
    plt.suptitle(f"THỰC CHỨNG 8 CA BẪY ĐẠO CỤ RỖNG U8 (GROUND TRUTH = 0)\nĐánh giá hiện tượng phanh giật cục trên Robot Dẫn Đường Bệnh Viện",
                 fontsize=14, fontweight="bold", y=0.98)
    plt.tight_layout()
    plt.show()

render_u8_empty_traps_gallery(model_name="yolov8n.pt", conf_thresh=0.25, frame_step=1)


---
## 📊 Thống Kê Chi Tiết Hiệu Năng Thị Giác Trên Từng Đạo Cụ (Props Audit Table)
Duyệt qua toàn bộ **27 ca đạo cụ (486 frames)** để tính toán:
1. **Recall / TPR**: Tỷ lệ phát hiện người khi bị che khuất sau từng loại đạo cụ ($U_3$).
2. **False Positive Rate (FPR)**: Tỷ lệ báo động giả trên từng loại bẫy đạo cụ rỗng ($U_8$).
3. **Phát hiện xe lăn & cáng thương** ($U_4$).


In [ ]:
def run_prop_cases_statistical_audit(model_name="yolov8n.pt", conf_thresh=0.25):
    print(f"⏳ Đang thực hiện thống kê 486 frames đạo cụ cho mô hình: {model_name}...")
    detector = get_detector(model_name, conf_thresh)
    
    case_stats = {}
    for g_key, g_info in PROP_GROUPS.items():
        for cid, cname, prop_type in g_info["cases"]:
            case_stats[cid] = {
                "group": g_key,
                "name": cname,
                "prop_type": prop_type,
                "total_frames": 0,
                "gt_boxes": 0,
                "pred_boxes": 0,
                "tp": 0,
                "fp": 0,
                "fn": 0,
                "empty_tn": 0,
                "empty_fp": 0
            }
            
            for step in range(1, 19):
                fid = f"{cid.replace('.', '_')}_{step:04d}"
                img_p = img_dir / f"{fid}.jpg"
                if not img_p.exists():
                    continue
                    
                meta = annotations_registry.get(fid, {})
                gts = meta.get("ground_truth_annotations", [])
                gt_boxes = [g["bbox_xyxy_px"] for g in gts if len(g.get("bbox_xyxy_px", [])) == 4]
                is_trap = meta.get("is_hard_negative", False) or (len(gt_boxes) == 0)
                
                img_bgr = cv2.imread(str(img_p))
                h, w = img_bgr.shape[:2]
                raw_dets, _ = detector.detect(img_bgr)
                
                preds = []
                for d in raw_dets:
                    if d.get("class_id") == 0:
                        preds.append([d["x1"] * w, d["y1"] * h, d["x2"] * w, d["y2"] * h])
                        
                case_stats[cid]["total_frames"] += 1
                case_stats[cid]["gt_boxes"] += len(gt_boxes)
                case_stats[cid]["pred_boxes"] += len(preds)
                
                if is_trap:
                    if len(preds) > 0:
                        case_stats[cid]["empty_fp"] += 1
                    else:
                        case_stats[cid]["empty_tn"] += 1
                else:
                    matched = set()
                    for pb in preds:
                        best_iou = 0.0
                        best_g = -1
                        for gi, gb in enumerate(gt_boxes):
                            if gi in matched: continue
                            iou = compute_iou(pb, gb)
                            if iou > best_iou:
                                best_iou = iou
                                best_g = gi
                        if best_iou >= 0.35 and best_g != -1:
                            case_stats[cid]["tp"] += 1
                            matched.add(best_g)
                        else:
                            case_stats[cid]["fp"] += 1
                    case_stats[cid]["fn"] += (len(gt_boxes) - len(matched))
                    
    # In bảng kết quả
    print(f"\n{'='*95}")
    print(f"{'Mã Ca':<8} | {'Tên Đạo Cụ Lâm Sàng':<35} | {'Frames':<6} | {'GT Box':<6} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'TPR %':<7} | {'FPR Bẫy %':<9}")
    print(f"{'='*95}")
    
    for cid, st in case_stats.items():
        tpr_str = f"{st['tp']/st['gt_boxes']*100:.1f}%" if st['gt_boxes'] > 0 else "N/A (Trap)"
        total_empty = st['empty_fp'] + st['empty_tn']
        fpr_str = f"{st['empty_fp']/total_empty*100:.1f}%" if total_empty > 0 else "0.0%"
        print(f"{cid:<8} | {st['name']:<35} | {st['total_frames']:<6} | {st['gt_boxes']:<6} | {st['tp']:<5} | {st['fp']:<5} | {st['fn']:<5} | {tpr_str:<7} | {fpr_str:<9}")
        
    print(f"{'='*95}")
    return case_stats

stats_v8 = run_prop_cases_statistical_audit("yolov8n.pt", 0.25)


In [ ]:
# Vẽ đồ thị phân tích khả năng chống báo động giả (False Alarm Rate) trên từng bẫy đạo cụ U8
u8_labels = []
u8_fprs = []

for cid, st in stats_v8.items():
    if cid.startswith("U8"):
        tot = st["empty_fp"] + st["empty_tn"]
        fpr = (st["empty_fp"] / tot * 100.0) if tot > 0 else 0.0
        u8_labels.append(f"[{cid}] {st['name'].replace('Bẫy ', '')}")
        u8_fprs.append(fpr)

fig, ax = plt.subplots(figsize=(12, 5), dpi=110)
colors = ["#d9534f" if v > 50 else "#f0ad4e" if v > 0 else "#5cb85c" for v in u8_fprs]
bars = ax.barh(u8_labels, u8_fprs, color=colors, edgecolor="black", height=0.6)

for bar, v in zip(bars, u8_fprs):
    ax.text(v + 1.5, bar.get_y() + bar.get_height()/2, f"{v:.1f}%", va="center", fontsize=9, fontweight="bold")

ax.set_xlim(0, 115)
ax.set_xlabel("Tỷ Lệ Báo Động Giả / Phanh Nhầm Robot (False Positive Rate %)", fontsize=11, fontweight="bold")
ax.set_title("ĐÁNH GIÁ ĐỘ DỄ BỊ LỪA CỦA YOLO TRÊN CÁC ĐẠO CỤ Y TẾ RỖNG (U8 EMPTY TRAPS)", fontsize=12, fontweight="bold")
ax.grid(axis="x", linestyle="--", alpha=0.6)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


---
## 🎯 Kết Luận & Khuyến Nghị Tích Hợp Hệ Thống Robot Nav2
1. **Vấn đề Báo động giả trên Standee 2D (`U8.8`) và Kiosk (`U8.4`)**:
   * Các mô hình AI Pretrained (COCO) có tỷ lệ phanh nhầm cực cao đối với Standee in hình người thật do thiếu nhận biết chiều sâu 3D.
   * **Giải pháp kiến trúc**: Kết hợp Camera RGB với thông tin độ sâu Depth (RGB-D) hoặc lọc qua bản đồ tĩnh Nav2 Static Layer để loại trừ Standee và Kiosk cố định.
2. **Khả năng phát hiện người sau vật cản mỏng (`U3.1` Cột truyền dịch, `U3.10` Xe tiêm)**:
   * Thuật toán dán nhãn 128 tia Multi-Raycast đã khắc phục triệt để lỗi sinh hộp bao ảo bao trùm toàn bộ xe cản. AI duy trì tỷ lệ nhận diện cao đối với phần thân người lộ diện.
3. **Xe lăn và Cáng thương (`U4.1`, `U4.6`)**:
   * Mô hình cần được gắn cờ vùng ưu tiên an toàn (Safety Critical Zone) để robot chủ động giảm tốc độ khi di chuyển qua các hành lang có mật độ phương tiện y tế cao.
